In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pyprojroot import here
import os
import importlib

In [2]:
gammaval=0.5

In [3]:
colors=["#0091ff",'#f2d138', '#7fc960', "#a589ee"] #majority/minority
maj_color = colors[0]
min_color = colors[1]
min_maj_color = colors[2]
maj_min_color = colors[3]
greycolor='#626262'
lightgreycolor='#c4c4c4'
myblack='#222222'


SMALL_SIZE = 8
MEDIUM_SIZE = 9
BIGGER_SIZE = 12

# --- FONT SIZES ----------------------------------------------------------
TITLE_SIZE = 9
LABEL_SIZE = 8
TICK_SIZE = 8
ANNOT_SIZE = 7
LEGEND_SIZE = 7

linewidth=0.5
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

def customize_axes(ax,myblack):
    ax.spines['top'].set_color(myblack)
    ax.spines['bottom'].set_color(myblack)
    ax.spines['left'].set_color(myblack)
    ax.spines['right'].set_color(myblack)
    ax.xaxis.label.set_color(myblack)
    ax.yaxis.label.set_color(myblack)
    ax.title.set_color(myblack)
    ax.tick_params(axis='x', colors=myblack)
    ax.tick_params(axis='y', colors=myblack)

In [4]:

src_path = os.path.join(here(), "03_analytical", "src", "functions.py")

spec = importlib.util.spec_from_file_location("fun", src_path)
fun = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fun)


In [5]:
## define model parameters

No = 333 # number of opponents
Ns = 1000-No  # number of supporters
N = No + Ns # total number of agents
fs = Ns / N # fraction of supporters
fo = No / N # fraction of opponents


hss = 0.5 # homophily supporter-supporter
hso = 1 - hss # homophily supporter-opponent, following Karimi et al. (2018)
hoo = 0.5 # homophily opponent-opponent
hos = 1 - hoo # homophily opponent-supporter

In [6]:
def G1_orig(H_in, H_out, min_fraction, eps=1e-10):
    maj_fraction = 1 - min_fraction
    den = np.maximum(maj_fraction * H_in + min_fraction * H_out, eps)
    return maj_fraction - (maj_fraction * H_in) / den

def G2_orig(H_in, H_out, min_fraction, eps=1e-10):
    maj_fraction = 1 - min_fraction
    den = np.maximum(maj_fraction * H_out + min_fraction * H_in, eps)
    return maj_fraction - (maj_fraction * H_out) / den

def G1_bayes(H_in, H_out, min_fraction, delta, gamma, eps=1e-10):
    maj_fraction = 1 - min_fraction
    den = np.maximum(maj_fraction * H_in + min_fraction * H_out, eps)
    return maj_fraction - fun.rescale_bayes((maj_fraction * H_in) / den, gamma=gamma, delta=delta)

def G2_bayes(H_in, H_out, min_fraction, delta, gamma, eps=1e-10):
    maj_fraction = 1 - min_fraction
    den = np.maximum(maj_fraction * H_out + min_fraction * H_in, eps)
    return maj_fraction - fun.rescale_bayes((maj_fraction * H_out) / den, gamma=gamma, delta=delta)

In [7]:
# --- 3-panel heatmaps for beta_o, beta_s, beta_overall with contours ---
n_points = 101
h_in_supp_vals = np.linspace(0, 1, n_points)
h_in_opp_vals = np.linspace(0, 1, n_points)
h_out = 0.3
# Containers
mis_o_SBM = np.zeros((n_points, n_points))
mis_s_SBM = np.zeros((n_points, n_points))
mis_overall_SBM = np.zeros((n_points, n_points))

for i, h_in_opp in enumerate(h_in_opp_vals):
    for j, h_in_supp in enumerate(h_in_supp_vals):
        # Misperceptions
        beta_s = G1_orig(h_in_supp, h_out, fo, eps=1e-10)
        beta_o = G2_orig(h_in_opp, h_out, fo, eps=1e-10)
        beta_overall = fun.compute_overall_misperception(fs, beta_o, beta_s)

        mis_o_SBM[i, j] = beta_o
        mis_s_SBM[i, j] = beta_s
        mis_overall_SBM[i, j] = beta_overall

print(np.max(mis_o_SBM))
print(np.max(mis_s_SBM))
print(np.max(mis_overall_SBM))
print(np.argmax(mis_overall_SBM))

0.29164828362408557
0.667
0.5420078784468205
10100


In [8]:
# --- SBM grid + Bayesian rescaling, varying delta ---
n_points = 101
h_in_supp_vals = np.linspace(0, 1, n_points)
h_in_opp_vals  = np.linspace(0, 1, n_points)
h_out = 0.3

delta_vals = [0.5, 1, 2]
gamma = gammaval

# --- Predefined bayes functions ---
def G1_bayes(H_in, H_out, min_fraction, delta, gamma, eps=1e-10):
    # H_in for majority group (supporters if fs is majority)
    maj_fraction = 1 - min_fraction
    den = maj_fraction * H_in + min_fraction * H_out
    den = np.maximum(den, eps)
    return maj_fraction - fun.rescale_bayes((maj_fraction * H_in) / den, gamma=gamma, delta=delta)

def G2_bayes(H_in, H_out, min_fraction, delta, gamma, eps=1e-10):
    # H_in for minority group (opponents if fs is majority)
    maj_fraction = 1 - min_fraction
    den = maj_fraction * H_out + min_fraction * H_in
    den = np.maximum(den, eps)
    return maj_fraction - fun.rescale_bayes((maj_fraction * H_out) / den, gamma=gamma, delta=delta)

# Containers per-delta
mis_o_SBM_by_delta = {}
mis_s_SBM_by_delta = {}
mis_overall_SBM_by_delta = {}

for delta in delta_vals:
    mis_o_SBM = np.zeros((n_points, n_points))
    mis_s_SBM = np.zeros((n_points, n_points))
    mis_overall_SBM = np.zeros((n_points, n_points))

    for i, h_in_opp in enumerate(h_in_opp_vals):
        for j, h_in_supp in enumerate(h_in_supp_vals):
            beta_s = G1_bayes(h_in_supp, h_out, fo, delta=delta, gamma=gamma, eps=1e-10)
            beta_o = G2_bayes(h_in_opp,  h_out, fo, delta=delta, gamma=gamma, eps=1e-10)
            beta_overall = fun.compute_overall_misperception(fs, beta_o, beta_s)

            mis_o_SBM[i, j] = beta_o
            mis_s_SBM[i, j] = beta_s
            mis_overall_SBM[i, j] = beta_overall

    mis_o_SBM_by_delta[delta] = mis_o_SBM
    mis_s_SBM_by_delta[delta] = mis_s_SBM
    mis_overall_SBM_by_delta[delta] = mis_overall_SBM

    print(f"\n--- delta = {delta}, gamma = {gamma} ---")
    print("max beta_o:", np.max(mis_o_SBM))
    print("max beta_s:", np.max(mis_s_SBM))
    print("max beta_overall:", np.max(mis_overall_SBM))
    imax = np.unravel_index(np.argmax(mis_overall_SBM), mis_overall_SBM.shape)
    print("argmax (h_in_opp, h_in_supp):",
          (h_in_opp_vals[imax[0]], h_in_supp_vals[imax[1]]))


--- delta = 0.5, gamma = 0.5 ---
max beta_o: 0.3129390816623346
max beta_s: 0.667
max beta_overall: 0.5490977141935575
argmax (h_in_opp, h_in_supp): (np.float64(1.0), np.float64(0.0))

--- delta = 1, gamma = 0.5 ---
max beta_o: 0.23032379692914062
max beta_s: 0.667
max beta_overall: 0.5215868243774039
argmax (h_in_opp, h_in_supp): (np.float64(1.0), np.float64(0.0))

--- delta = 2, gamma = 0.5 ---
max beta_o: 0.14403842043926063
max beta_s: 0.667
max beta_overall: 0.4928537940062738
argmax (h_in_opp, h_in_supp): (np.float64(1.0), np.float64(0.0))


In [9]:
# --- Build mis_overall_by_delta for BA (Bayesian rescaling), using your style/template ---
n_points = 101
hss_vals = np.linspace(0, 1, n_points)
hoo_vals = np.linspace(0, 1, n_points)

delta_vals = [0.5, 1, 2]
gamma = gammaval  # fixed uncertainty

# Store overall misperception for each delta
mis_overall_by_delta = {}

for delta in delta_vals:
    mis_overall = np.zeros((n_points, n_points))

    for i, hoo in enumerate(hoo_vals):
        for j, hss in enumerate(hss_vals):
            hso = 1 - hss
            hos = 1 - hoo

            # Solve for C
            C = fun.compute_C(fo, fs, hoo, hos, hso, hss)

            # Probabilities
            p_oo = fun.compute_p_oo(hoo, hos, C)
            p_ss = fun.compute_p_ss(hss, hso, C)
            p_os = fun.compute_p_os(hos, hoo, C)
            p_so = fun.compute_p_so(hso, hss, C)

            # Misperceptions -> overall
            beta_o = fun.compute_misperception(
                fs,
                fun.rescale_bayes(
                    fun.compute_opponent_perception(No, Ns, p_os, p_so, p_oo),
                    gamma=gamma, delta=delta
                )
            )
            beta_s = fun.compute_misperception(
                fs,
                fun.rescale_bayes(
                    fun.compute_supporter_perception(No, Ns, p_os, p_so, p_ss),
                    gamma=gamma, delta=delta
                )
            )
            beta_overall = fun.compute_overall_misperception(fs, beta_o, beta_s)

            mis_overall[i, j] = beta_overall

    mis_overall_by_delta[delta] = mis_overall

for d in delta_vals:
    arr = mis_overall_by_delta[d]
    print(f"delta={d}: min={arr.min():.3f}, max={arr.max():.3f}")

delta=0.5: min=-0.057, max=0.529
delta=1: min=-0.111, max=0.500
delta=2: min=-0.159, max=0.472


In [ ]:
# --- 2x3 panel heatmaps: beta_overall varying delta (BA top, SBM bottom) ---

delta_vals = [0.5, 1, 2]
gamma = gammaval

# --- Shared color scale across ALL panels (BA + SBM, all deltas) ---
vmax = max(
    max(abs(mis_overall_by_delta[d]).max() for d in delta_vals),
    max(abs(mis_overall_SBM_by_delta[d]).max() for d in delta_vals),
)
vmin = -vmax

# --- Figure and axes: 2 rows x 3 columns ---
fig, axes = plt.subplots(2, 3, figsize=(7.5, 4.8), constrained_layout=False)
plt.subplots_adjust(hspace=0.6)

tick_vals = np.linspace(0, 1, 5)

# Grids for contours (precompute once)
X_ba, Y_ba = np.meshgrid(hss_vals, hoo_vals)
X_sbm, Y_sbm = np.meshgrid(h_in_supp_vals, h_in_opp_vals)
levels = [0.66 - 0.41, 0.66 - 0.36]

panel_labels = ["C", "D", "E", "F", "G", "H"]

ims = []

for col, delta in enumerate(delta_vals):
    # =======================
    # Top row: BA
    # =======================
    ax = axes[0, col]
    data_ba = mis_overall_by_delta[delta]

    im = ax.imshow(
        data_ba,
        extent=[0, 1, 0, 1],
        origin="lower",
        cmap="RdBu_r",
        vmin=vmin, vmax=vmax,
        aspect="equal",
    )
    ims.append(im)

    ax.set_xticks(tick_vals)
    ax.set_yticks(tick_vals)

    if col == 0:
        ax.set_ylabel(r"$h_{oo}$")
    else:
        ax.set_ylabel("")
    ax.set_xlabel(r"$h_{ss}$")

    ax.set_title(rf"{panel_labels[col]}. $\beta_{{\mathrm{{overall}}}}$ ($\delta={delta}$)")

    ctr = ax.contour(
        X_ba, Y_ba, data_ba,
        levels=levels,
        colors=[myblack],
        linewidths=1.2
    )
    ax.clabel(ctr, inline=True, fmt="%.2f", fontsize=8, colors=[myblack])

    # Zero contour on BA panel (dotted, labeled)
    zero_ctr = ax.contour(X_ba, Y_ba, data_ba, levels=[0], colors=[myblack], linewidths=1.2, linestyles='dotted')
    ax.clabel(zero_ctr, inline=True, fmt="%.2f", fontsize=8, colors=[myblack])

    # =======================
    # Bottom row: SBM
    # =======================
    ax2 = axes[1, col]
    data_sbm = mis_overall_SBM_by_delta[delta]

    im2 = ax2.imshow(
        data_sbm,
        extent=[0, 1, 0, 1],
        origin="lower",
        cmap="RdBu_r",
        vmin=vmin, vmax=vmax,
        aspect="equal",
    )
    ims.append(im2)

    ax2.set_xticks(tick_vals)
    ax2.set_yticks(tick_vals)

    if col == 0:
        ax2.set_ylabel(r"$H_{\text{in}}^o$")
    else:
        ax2.set_ylabel("")
    ax2.set_xlabel(r"$H_{\text{in}}^s$")

    ax2.set_title(rf"{panel_labels[col+3]}. $\beta_{{\mathrm{{overall}}}}$ ($\delta={delta}$)")

    ctr2 = ax2.contour(
        X_sbm, Y_sbm, data_sbm,
        levels=levels,
        colors=[myblack],
        linewidths=1.2
    )
    ax2.clabel(ctr2, inline=True, fmt="%.2f", fontsize=8, colors=[myblack])

    # Zero contour on SBM panel (dotted, labeled)
    zero_ctr2 = ax2.contour(X_sbm, Y_sbm, data_sbm, levels=[0], colors=[myblack], linewidths=1.2, linestyles='dotted')
    ax2.clabel(zero_ctr2, inline=True, fmt="%.2f", fontsize=8, colors=[myblack])

# --- Shared colorbar underneath ---
cbar_ax = fig.add_axes([0.15, -0.02, 0.70, 0.02])
cbar = fig.colorbar(ims[-1], cax=cbar_ax, orientation="horizontal")
cbar.set_label(r"Overall misperception $\beta_{\mathrm{overall}}$")

# --- Row labels ---
fig.text(
    0.5, 0.945,
    rf"Rescaled misperception on preferential attachment networks ($\gamma={gamma}$)",
    ha="center", va="center", fontsize=TITLE_SIZE
)
fig.text(
    0.5, 0.47,
    rf"Rescaled misperception on stochastic block networks ($\gamma={gamma}$, fixed $H_{{\text{{out}}}}={h_out}$)",
    ha="center", va="center", fontsize=TITLE_SIZE
)

plt.savefig(here("figures/figure_bayesian_rescaling_SBM_BA.pdf"), dpi=300, bbox_inches="tight")
plt.show()

**Figure. Overall misperception $\beta_{\mathrm{overall}}$ under Bayesian rescaling, for preferential attachment (top) and stochastic block (bottom) networks, varying the prior $\delta$ ($\gamma=0.5$).** Each panel is a heatmap of overall misperception $\beta_{\mathrm{overall}} = f_s - \hat{f}_s$ over a grid of within-group homophily parameters, with red indicating overestimation of the majority and blue indicating underestimation. **Top row (C–E):** Barabási–Albert preferential attachment networks; x-axis is supporter homophily $h_{ss}$, y-axis is opponent homophily $h_{oo}$. **Bottom row (F–H):** Stochastic block model networks with fixed out-group connection probability $H_{\mathrm{out}}=0.3$; x-axis is $H_{\mathrm{in}}^s$ (supporter in-group rate), y-axis is $H_{\mathrm{in}}^o$ (opponent in-group rate). Columns vary the Bayesian prior $\delta \in \{0.5, 1, 2\}$, with $\delta < 1$ reflecting a prior favouring the minority, $\delta = 1$ a neutral prior, and $\delta > 1$ a prior favouring the majority. Contour lines are drawn at $\beta_{\mathrm{overall}} \in \{0.25, 0.30\}$. The population share of supporters is fixed at $f_s = 2/3$.

## Robustness check: varying $H_{\text{out}}$ in the SBM

The main figure fixes $H_{\text{out}} = 0.3$. Here we show that the qualitative pattern — limited misperception range in the SBM regardless of Bayesian rescaling — holds across a range of $H_{\text{out}}$ values.

In [11]:
# --- Robustness check: SBM misperception for varying H_out, delta=1 ---
n_points = 101
h_in_supp_vals = np.linspace(0, 1, n_points)
h_in_opp_vals  = np.linspace(0, 1, n_points)

h_out_vals = [0.1, 0.3, 0.5, 0.8]
delta_rob  = 0.5
gamma_rob  = 0.5

mis_s_SBM_by_hout       = {}
mis_o_SBM_by_hout       = {}
mis_overall_SBM_by_hout = {}

for h_out_val in h_out_vals:
    mis_s       = np.zeros((n_points, n_points))
    mis_o       = np.zeros((n_points, n_points))
    mis_overall = np.zeros((n_points, n_points))
    for i, h_in_opp in enumerate(h_in_opp_vals):
        for j, h_in_supp in enumerate(h_in_supp_vals):
            beta_s = G1_bayes(h_in_supp, h_out_val, fo, delta=delta_rob, gamma=gamma_rob)
            beta_o = G2_bayes(h_in_opp,  h_out_val, fo, delta=delta_rob, gamma=gamma_rob)
            mis_s[i, j]       = beta_s
            mis_o[i, j]       = beta_o
            mis_overall[i, j] = fun.compute_overall_misperception(fs, beta_o, beta_s)
    mis_s_SBM_by_hout[h_out_val]       = mis_s
    mis_o_SBM_by_hout[h_out_val]       = mis_o
    mis_overall_SBM_by_hout[h_out_val] = mis_overall
    print(f"H_out={h_out_val}: max overall={mis_overall.max():.3f}, max beta_s={mis_s.max():.3f}, max beta_o={mis_o.max():.3f}")

H_out=0.1: max overall=0.587, max beta_s=0.667, max beta_o=0.427
H_out=0.3: max overall=0.549, max beta_s=0.667, max beta_o=0.313
H_out=0.5: max overall=0.529, max beta_s=0.667, max beta_o=0.253
H_out=0.8: max overall=0.510, max beta_s=0.667, max beta_o=0.195


In [ ]:
# --- Plot: 3x4 heatmaps (rows: overall, supporters, opponents; cols: H_out values) ---
row_data   = [mis_overall_SBM_by_hout, mis_s_SBM_by_hout, mis_o_SBM_by_hout]
row_titles = [r"Overall $\beta_{\mathrm{overall}}$",
              r"Supporters $\beta_s$",
              r"Opponents $\beta_o$"]
row_ylabels = [r"$H_{\mathrm{in}}^o$", r"$H_{\mathrm{in}}^o$", r"$H_{\mathrm{in}}^o$"]

vmax_rob = max(
    abs(d[h]).max()
    for d in row_data
    for h in h_out_vals
)
vmin_rob = -vmax_rob

fig, axes = plt.subplots(3, 4, figsize=(10, 7.5), constrained_layout=True)

tick_vals = np.linspace(0, 1, 5)
X_sbm, Y_sbm = np.meshgrid(h_in_supp_vals, h_in_opp_vals)
levels = [0.66 - 0.41, 0.66 - 0.36]
col_labels = ["A", "B", "C", "D"]

ims = []
for row_idx, (data_dict, row_title, ylabel) in enumerate(zip(row_data, row_titles, row_ylabels)):
    for col_idx, h_out_val in enumerate(h_out_vals):
        ax = axes[row_idx, col_idx]
        data = data_dict[h_out_val]

        im = ax.imshow(
            data,
            extent=[0, 1, 0, 1],
            origin="lower",
            cmap="RdBu_r",
            vmin=vmin_rob, vmax=vmax_rob,
            aspect="equal",
        )
        ims.append(im)

        ax.set_xticks(tick_vals)
        ax.set_yticks(tick_vals)

        if col_idx == 0:
            ax.set_ylabel(f"{row_title}\n{ylabel}", color=myblack)
        else:
            ax.set_yticklabels([])

        if row_idx == 2:
            ax.set_xlabel(r"$H_{\mathrm{in}}^s$", color=myblack)

        if row_idx == 0:
            ax.set_title(rf"{col_labels[col_idx]}. $H_{{\mathrm{{out}}}}={h_out_val}$", color=myblack)

        customize_axes(ax, myblack)

        # Contours on overall row only
        if row_idx == 0:
            ctr = ax.contour(
                X_sbm, Y_sbm, data,
                levels=levels,
                colors=[myblack],
                linewidths=1.2
            )
            ax.clabel(ctr, inline=True, fmt="%.2f", fontsize=ANNOT_SIZE, colors=[myblack])

            # Zero contour (dotted, labeled)
            zero_ctr = ax.contour(
                X_sbm, Y_sbm, data,
                levels=[0],
                colors=[myblack],
                linewidths=1.2,
                linestyles='dotted'
            )
            ax.clabel(zero_ctr, inline=True, fmt="%.2f", fontsize=ANNOT_SIZE, colors=[myblack])

cbar = fig.colorbar(ims[0], ax=axes, fraction=0.03, pad=0.04)
cbar.set_label(r"Misperception $\beta$", color=myblack)
cbar.ax.yaxis.set_tick_params(color=myblack)
plt.setp(cbar.ax.get_yticklabels(), color=myblack)

fig.suptitle(
    rf"SBM misperception varying $H_{{\mathrm{{out}}}}$ ($\delta={delta_rob}$, $\gamma={gamma_rob}$)",
    color=myblack, fontsize=TITLE_SIZE
)

plt.savefig(here("figures/figure_robustness_hout.pdf"), dpi=300, bbox_inches="tight")
plt.show()